# Project 86 — Local Chart Understanding Agent
## Data Table → Chart Synthesis → Insight Extraction

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Simulated Chart Data

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

charts = [
    {
        "type": "bar_chart", "title": "Monthly Revenue by Product",
        "data": {"Product A": [120, 135, 150, 148, 165, 180],
                 "Product B": [80, 85, 90, 95, 110, 105],
                 "Product C": [45, 50, 55, 60, 65, 70]},
        "x_axis": "Months (Jan-Jun)", "y_axis": "Revenue ($K)",
    },
    {
        "type": "line_chart", "title": "User Growth Trend",
        "data": {"DAU": [5200, 5800, 6100, 7200, 8500, 9100, 10200, 11500],
                 "MAU": [15000, 16500, 17800, 19200, 21000, 23500, 26000, 28500]},
        "x_axis": "Months (Jan-Aug)", "y_axis": "Users",
    },
    {
        "type": "pie_chart", "title": "Customer Segments",
        "data": {"Enterprise": 35, "Mid-Market": 28, "SMB": 22, "Startup": 15},
        "note": "By revenue contribution (%)",
    },
]
print(f"Charts to analyze: {len(charts)}")
for c in charts:
    print(f"  {c['type']}: {c['title']}")

## Step 2 — Structured Chart Analysis

In [ ]:
class DataTrend(BaseModel):
    direction: str = Field(description="increasing, decreasing, stable, fluctuating")
    growth_rate: str = Field(description="percentage or description")
    notable_points: list[str]

class ChartInsight(BaseModel):
    chart_type: str
    title: str
    key_finding: str
    trends: list[DataTrend]
    comparisons: list[str]
    anomalies: list[str]
    business_implication: str
    confidence: float = Field(ge=0, le=1)

analyzer = llm.with_structured_output(ChartInsight)

analyses = []
for chart in charts:
    insight = analyzer.invoke(
        f"Analyze this chart:\n"
        f"Type: {chart['type']}\nTitle: {chart['title']}\n"
        f"Data: {json.dumps(chart['data'])}\n"
        f"X-axis: {chart.get('x_axis','')} | Y-axis: {chart.get('y_axis','')}"
    )
    analyses.append(insight)
    print(f"\n{insight.title}:")
    print(f"  Finding: {insight.key_finding}")
    print(f"  Trends: {len(insight.trends)} detected")
    print(f"  Anomalies: {insight.anomalies[:2]}")
    print(f"  Implication: {insight.business_implication[:80]}")

## Step 3 — Executive Summary Generation

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a 3-paragraph executive summary of these chart analyses. "
     "Use specific numbers. Highlight risks and opportunities."),
    ("human", "Chart analyses:\n{analyses}")
])
summary_chain = summary_prompt | llm | StrOutputParser()

exec_summary = summary_chain.invoke({
    "analyses": json.dumps([a.model_dump() for a in analyses], indent=2)
})
print("EXECUTIVE SUMMARY")
print("=" * 50)
print(exec_summary[:600])

## Step 4 — Recommendation Engine

In [ ]:
rec_prompt = ChatPromptTemplate.from_messages([
    ("system", "Based on chart data, provide 5 specific, actionable recommendations. "
     "Each should reference specific data points."),
    ("human", "Data:\n{data}\n\nInsights:\n{insights}\n\n5 Recommendations:")
])
rec_chain = rec_prompt | llm | StrOutputParser()

recommendations = rec_chain.invoke({
    "data": json.dumps([c["data"] for c in charts], default=str),
    "insights": json.dumps([a.key_finding for a in analyses]),
})
print("RECOMMENDATIONS")
print("=" * 50)
print(recommendations[:500])

## What You Learned
- **Data-to-insight extraction** from chart representations
- **Trend detection** with direction and rate
- **Executive summary** generation from multiple charts
- **Actionable recommendations** grounded in data